# 01 — Adoption ingest: Entra sign-in logs → Bronze JSON

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 12 — T5 (finished in the Sprint 12.1 mini-sprint) |
| **Layer** | `Files/Bronze/adoption/YYYY-MM-DD/signins.json` (raw JSON landing) |
| **Source** | Entra sign-in logs — `SigninLogs` in `log-ihzhhpf-sit` (Log Analytics), routed by `infra/modules/entra/adoption-telemetry.bicep` |
| **Target** | `lh_ihzhhpf_sit` — `Files/Bronze/adoption/<day>/signins.json` (one file per sign-in day) |
| **Contract** | `docs/superpowers/specs/2026-07-09-sprint-12-org-design.md` §7 |
| **Consumers** | `data-platform/notebooks/bva/ingest_bronze_adoption.py` (Sprint 15 BVA) |

Reads the last 24h of `ihzhhpf-app` sign-ins and lands them **verbatim as JSON**
under `Files/Bronze/adoption/<day>/signins.json`, one file per sign-in day. The
row shape is the Bronze adoption contract (§7) — **identical** to the synthetic
backfill produced by `data-platform/scripts/adoption_seed_synthetic.py`, so real
and seeded telemetry are interchangeable for the downstream BVA medallion. A
downstream **Silver** notebook does the deduplication + role/hospital join.

**No PHI** — sign-in metadata carries UPN + IP only; the IP is redacted to a
`/24` in Bronze. Scheduled nightly by `.github/workflows/adoption-refresh.yml`.

> **Source note.** This notebook reads from Log Analytics because
> `adoption-telemetry.bicep` already routes `SigninLogs` there. The same Bronze
> rows can be produced from the Microsoft Graph `auditLogs/signIns` endpoint
> (used read-only by the `onboarding-agent`): keep the same projection field
> names and the pure `adoption_transforms` mapping below is unchanged.

> When real telemetry has not yet accumulated, seed Bronze with
> `python3 data-platform/scripts/adoption_seed_synthetic.py --output-dir Files/Bronze --days 30`.


In [ ]:
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
target_lakehouse = 'lh_ihzhhpf_sit'
log_analytics_workspace = 'log-ihzhhpf-sit'  # linked service name; resolved to a workspace GUID at runtime
bronze_root = 'Files/Bronze/adoption'        # lakehouse-relative Bronze landing folder
personas_path = 'Files/reference/personas.csv'  # UPN -> app_role roster (mirrors data/synthetic/personas.csv)
lookback = '24h'                             # Kusto lookback window
default_env = 'sit'                          # fallback slot when the app URL host is not -prod


## 1. Kusto query

Projects the raw sign-in fields the Bronze contract needs (design spec §7). The
`env` slot and the `appRole` join are derived below (env from the app host,
`appRole` from the personas roster); the Kusto projection keeps the raw shape.


In [ ]:
KQL = f"""
SigninLogs
| where TimeGenerated > ago({lookback})
| where AppDisplayName startswith "ihzhhpf-app"
| project
    TimeGenerated,
    UserId,
    UserPrincipalName,
    AppDisplayName,
    AppId,
    ResultType,
    IPAddress,
    ClientAppUsed,
    DeviceDetail_TrustType = tostring(DeviceDetail.trustType),
    Location_CountryOrRegion = tostring(LocationDetails.countryOrRegion)
"""


## 2. Read from Log Analytics

Uses the Azure Monitor / Log Analytics Spark connector. Managed identity /
workload identity federation provides the token (no secrets in the notebook).


In [ ]:
df = (
    spark.read.format('com.microsoft.kusto.spark.synapse.datasource')
    .option('spark.synapse.linkedService', log_analytics_workspace)
    .option('query', KQL)
    .load()
)
df.printSchema()


## 3. Load the persona → app-role roster

The Bronze contract carries `appRole` so the Sprint 15 BVA adoption KPI can
attribute each sign-in to a product capability. The roster mirrors
`data/synthetic/personas.csv`; an unknown user gets `appRole = None` and is
attributed to `Aggregated` downstream.


In [ ]:
persona_role = {}
try:
    roster = spark.read.option('header', True).csv(personas_path)
    persona_role = {
        r['upn'].lower(): r['app_role']
        for r in roster.select('upn', 'app_role').collect()
        if r['upn']
    }
except Exception as exc:  # roster is optional — Bronze stays raw, Silver can join later
    print(f'WARN: could not load personas roster from {personas_path}: {exc}')
print(f'Loaded {len(persona_role)} persona role mappings')


## 4. Map to the Bronze contract and land one JSON file per sign-in day

`adoption_transforms` is the pure, unit-tested mapping (see
`data-platform/notebooks/adoption/tests/`). It redacts the IP to a `/24`,
derives `env` from the app host, joins `appRole`, and normalises the timestamp.
Bronze is append-only and idempotent per day: re-running overwrites the same
`<day>/signins.json`; Silver handles deduplication across days.


In [ ]:
import json
import notebookutils
from adoption_transforms import to_bronze_rows, group_by_signin_day

raw_rows = [row.asDict() for row in df.collect()]
bronze_rows = to_bronze_rows(raw_rows, persona_role, default_env=default_env)
by_day = group_by_signin_day(bronze_rows)

written = 0
for day, rows in sorted(by_day.items()):
    target = f'{bronze_root}/{day}/signins.json'
    payload = json.dumps(rows, indent=2, ensure_ascii=False)
    notebookutils.fs.put(target, payload, True)  # overwrite=True — idempotent per day
    written += len(rows)
    print(f'wrote {len(rows):>5d} rows -> {target}')

print('-' * 72)
print(f'Bronze adoption ingest: {written} sign-in rows across {len(by_day)} day(s) under {bronze_root}/')
